# 7. Process PyNNLF Output: South Australia (SA) battery energy storage system (BESS) 44hh Clean Cohort

## Purpose
Databricks full-run version. This notebook reads the complete 36-experiment recap from `experiment_result_databricks` and writes paper-facing CSVs to `results/04_sa_bess_clean_44hh`.

## Inputs
- `RECAP_PATH`

## Run Flow
1. Setup And Paths
2. Load And Validate Databricks Recap Rows
3. Build Tables

## Outputs
- `RESULTS_DIR / "sa_bess_44hh_fh8_combined_recap.csv"`
- `RESULTS_DIR / "sa_bess_44hh_fh8_nrmse_comparison.csv"`
- `RESULTS_DIR / "sa_bess_44hh_fh8_nrmse_stddev_comparison.csv"`
- `RESULTS_DIR / "paper_table_sa_bess_44hh_signal_test_nrmse.csv"`
- `RESULTS_DIR / "paper_table_sa_bess_44hh_key_models_train_test_runtime.csv"`

## 1. Setup And Paths

In [1]:
from pathlib import Path
import sys

def find_publication_project(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "specs").exists() and (candidate / "data").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not find publication/journal_article_1 from the current working directory.")

def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "pynnlf").exists():
            return candidate
    raise FileNotFoundError("Could not find the PyNNLF repo root.")
PROJECT_DIR = find_publication_project()
REPO_ROOT = find_repo_root(PROJECT_DIR)
print(f"Publication project: {PROJECT_DIR}")
print(f"Repository root: {REPO_ROOT}")
print("Experiment result source: Databricks complete run")
import pandas as pd
RESULTS_DIR = PROJECT_DIR / "results" / "04_sa_bess_clean_44hh"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENT_RESULTS_DIR = PROJECT_DIR / "experiment_result_databricks"
RECAP_PATH = EXPERIMENT_RESULTS_DIR / "a1_experiment_result.csv"
MODEL_ORDER = ['m1_naive_hp1', 'm2_snaive_hp2', 'm3_ets_hp1', 'm4_arima_hp1', 'm6_lr_hp1', 'm7_ann_hp1', 'm8_dnn_hp1', 'm9_rt_hp3', 'm10_rf_hp1', 'm13_lstm_hp2', 'm16_prophet_hp1', 'm17_xgb_hp1']
DATASET_LABELS = {"ds22": "underlying_load", "ds23": "net_load_with_pv", "ds24": "net_load_with_pv_battery"}
FH8_MINUTES = 1440

Publication project: <local path redacted>
Repository root: <local path redacted>
Experiment result source: Databricks complete run


## 2. Load And Validate Databricks Recap Rows

In [2]:
if not RECAP_PATH.exists():
    raise FileNotFoundError(f"Missing recap at {RECAP_PATH}. Run notebook 6 first.")
recap = pd.read_csv(RECAP_PATH)
required = {"dataset_no", "forecast_horizon_min", "model_name", "test_nRMSE", "test_nRMSE_stddev"}
missing = required - set(recap.columns)
if missing:
    raise ValueError(f"Recap is missing required columns: {sorted(missing)}")
rows = recap.loc[recap["dataset_no"].isin(DATASET_LABELS) & pd.to_numeric(recap["forecast_horizon_min"], errors="coerce").eq(FH8_MINUTES)].copy()
rows["dataset_label"] = rows["dataset_no"].map(DATASET_LABELS)
rows["result_source"] = "experiment_result_databricks"
rows["model_name"] = pd.Categorical(rows["model_name"], categories=MODEL_ORDER, ordered=True)
expected = {(ds_id, model) for ds_id in DATASET_LABELS for model in MODEL_ORDER}
actual = set(zip(rows["dataset_no"].astype(str), rows["model_name"].astype(str)))
missing_keys = sorted(expected - actual)
if missing_keys:
    raise ValueError(f"SA BESS 44hh results are incomplete. Missing {len(missing_keys)} combinations: {missing_keys[:10]}")
if rows.shape[0] != len(expected):
    raise ValueError(f"Expected {len(expected)} rows, found {rows.shape[0]}")
rows = rows.sort_values(["model_name", "dataset_label"])
display(rows[["dataset_no", "dataset_label", "model_name", "test_nRMSE", "test_nRMSE_stddev"]])

,dataset_no,dataset_label,model_name,test_nRMSE,test_nRMSE_stddev
12,ds23,net_load_with_pv,m1_naive_hp1,16.791714,2.690041
24,ds24,net_load_with_pv_battery,m1_naive_hp1,21.236350,3.984657
0,ds22,underlying_load,m1_naive_hp1,5.698227,1.183842
13,ds23,net_load_with_pv,m2_snaive_hp2,21.370636,2.657737
25,ds24,net_load_with_pv_battery,m2_snaive_hp2,27.709284,4.555416
1,ds22,underlying_load,m2_snaive_hp2,6.692483,1.535510
14,ds23,net_load_with_pv,m3_ets_hp1,16.925135,2.817907
26,ds24,net_load_with_pv_battery,m3_ets_hp1,21.297951,4.221313
2,ds22,underlying_load,m3_ets_hp1,5.579634,1.191013
15,ds23,net_load_with_pv,m4_arima_hp1,51.458077,7.854137


## 3. Build Tables

In [3]:
def wide_metric_table(frame, metric_column):
    table = frame.pivot_table(index="model_name", columns="dataset_label", values=metric_column, aggfunc="first", observed=False)
    table = table.reindex(index=MODEL_ORDER)
    table = table[[DATASET_LABELS[key] for key in ["ds22", "ds23", "ds24"]]]
    table.index.name = "model_hp"
    table.columns.name = None
    return table

PAPER_KEY_MODELS = ['m17_xgb_hp1', 'm1_naive_hp1', 'm4_arima_hp1']
PAPER_MODEL_LABELS = {'m1_naive_hp1': 'naive_hp1', 'm2_snaive_hp2': 'snaive_hp2', 'm3_ets_hp1': 'ets_hp1', 'm4_arima_hp1': 'arima_hp1', 'm6_lr_hp1': 'lr_hp1', 'm7_ann_hp1': 'ann_hp1', 'm8_dnn_hp1': 'dnn_hp1', 'm9_rt_hp3': 'rt_hp3', 'm10_rf_hp1': 'rf_hp1', 'm13_lstm_hp2': 'lstm_hp2', 'm16_prophet_hp1': 'prophet_hp1', 'm17_xgb_hp1': 'xgb_hp1'}

def short_model_name(model_name):
    return PAPER_MODEL_LABELS.get(str(model_name), str(model_name))

def get_single_value(frame, dataset_label, model_name, column):
    match = frame.loc[frame["dataset_label"].eq(dataset_label) & frame["model_name"].astype(str).eq(model_name), column]
    if match.empty:
        raise ValueError(f"Missing {column} for {dataset_label} / {model_name}")
    return pd.to_numeric(match.iloc[0], errors="coerce")

nrmse = wide_metric_table(rows, "test_nRMSE")
stddev = wide_metric_table(rows, "test_nRMSE_stddev")
paper_model_order_by_underlying = nrmse.sort_values("underlying_load", ascending=True).index.tolist()
paper_table2_style = nrmse.reindex(paper_model_order_by_underlying).rename(index=short_model_name).round(2)

paper_rows = []
for model in PAPER_KEY_MODELS:
    row = {"Model Name": short_model_name(model)}
    for dataset_key in ["ds22", "ds23", "ds24"]:
        label = DATASET_LABELS[dataset_key]
        row[f"{label} Train nRMSE (%)"] = get_single_value(rows, label, model, "train_nRMSE")
        row[f"{label} Test nRMSE (%)"] = get_single_value(rows, label, model, "test_nRMSE")
        row[f"{label} Training Time (s)"] = get_single_value(rows, label, model, "runtime_ms") / 1000.0
    paper_rows.append(row)
paper_table3_style = pd.DataFrame(paper_rows)
for column in paper_table3_style.columns.drop("Model Name"):
    paper_table3_style[column] = pd.to_numeric(paper_table3_style[column], errors="coerce").round(1)

rows.to_csv(RESULTS_DIR / "sa_bess_44hh_fh8_combined_recap.csv", index=False)
nrmse.to_csv(RESULTS_DIR / "sa_bess_44hh_fh8_nrmse_comparison.csv")
stddev.to_csv(RESULTS_DIR / "sa_bess_44hh_fh8_nrmse_stddev_comparison.csv")
paper_table2_style.to_csv(RESULTS_DIR / "paper_table_sa_bess_44hh_signal_test_nrmse.csv")
paper_table3_style.to_csv(RESULTS_DIR / "paper_table_sa_bess_44hh_key_models_train_test_runtime.csv", index=False)

display(nrmse.round(3))
display(stddev.round(3))
display(paper_table2_style)
display(paper_table3_style)

,underlying_load,net_load_with_pv,net_load_with_pv_battery
model_hp,,,
m1_naive_hp1,5.698,16.792,21.236
m2_snaive_hp2,6.692,21.371,27.709
m3_ets_hp1,5.580,16.925,21.298
m4_arima_hp1,10.471,51.458,57.678
m6_lr_hp1,4.786,14.761,18.947
m7_ann_hp1,5.459,15.339,19.595
m8_dnn_hp1,5.267,15.518,19.852
m9_rt_hp3,7.017,21.645,28.096
m10_rf_hp1,5.223,17.390,21.674


,underlying_load,net_load_with_pv,net_load_with_pv_battery
model_hp,,,
m1_naive_hp1,1.184,2.690,3.985
m2_snaive_hp2,1.536,2.658,4.555
m3_ets_hp1,1.191,2.818,4.221
m4_arima_hp1,1.752,7.854,12.286
m6_lr_hp1,1.004,1.890,3.210
m7_ann_hp1,1.202,1.669,2.954
m8_dnn_hp1,1.121,1.580,3.179
m9_rt_hp3,1.278,1.954,3.786
m10_rf_hp1,1.036,2.424,3.333


,underlying_load,net_load_with_pv,net_load_with_pv_battery
model_hp,,,
lr_hp1,4.79,14.76,18.95
xgb_hp1,4.91,15.85,20.43
rf_hp1,5.22,17.39,21.67
dnn_hp1,5.27,15.52,19.85
ann_hp1,5.46,15.34,19.59
ets_hp1,5.58,16.93,21.30
naive_hp1,5.70,16.79,21.24
snaive_hp2,6.69,21.37,27.71
prophet_hp1,6.71,20.82,27.02


,Model Name,underlying_load Train nRMSE (%),underlying_load Test nRMSE (%),underlying_load Training Time (s),net_load_with_pv Train nRMSE (%),net_load_with_pv Test nRMSE (%),net_load_with_pv Training Time (s),net_load_with_pv_battery Train nRMSE (%),net_load_with_pv_battery Test nRMSE (%),net_load_with_pv_battery Training Time (s)
0,xgb_hp1,1.8,4.9,25.4,4.3,15.9,15.3,5.4,20.4,15.0
1,naive_hp1,5.8,5.7,0.0,17.0,16.8,0.0,21.6,21.2,0.0
2,arima_hp1,3.7,10.5,1.5,10.4,51.5,2.3,12.1,57.7,1.9
